# 🌸 Nayari Dataset Builder  v3
**Run locally.** Scans `dataset/` (including subdirectories) for `.md`, `.txt`, `.pdf`, and `.json` files, applies a robust multi-format parser, deduplicates, quality-filters, and exports `nayari_dataset.json`.  Then auto-uploads to Kaggle.

| Feature | Details |
|---|---|
| Speaker formats | `Name:`, `**Name**:`, `> Name:`, `[Name]:` |
| Scene splitting | Blank-line gaps, `---END---`, `<end>`, `===` dividers |
| Auto-aliases | Unknown speakers scanned from filenames & content |
| Quality filter | Min 2 turns, min 10 chars/message |
| Deduplication | MD5 fingerprint on normalised content |
| OOC stripping | `[OOC: …]` and `(OOC …)` blocks removed |
| Source tagging | Every conversation records its origin file |
| Token estimate | Rough GPT-style token count in the stats cell |


## 📦 Step 0 — Install & Import

In [4]:
%pip install pdfplumber kaggle -q

import re, json, os, shutil, subprocess, sys, hashlib, warnings
import pdfplumber
from pathlib import Path
from collections import defaultdict

warnings.filterwarnings("ignore", message="Could not get FontBBox")

DATASET_DIR = Path("dataset")
OUTPUT_FILE = Path("nayari_dataset.json")

# ── pretty directory scan ────────────────────────────────────────────────────
all_files = sorted(
    f for f in DATASET_DIR.rglob("*")
    if f.is_file() and f.suffix.lower() in {".md", ".txt", ".pdf", ".json"}
)
print(f"Found {len(all_files)} source file(s) in dataset/:\n")
for f in all_files:
    rel = f.relative_to(DATASET_DIR)
    print(f"  [{f.suffix.upper():5}] {rel}  ({f.stat().st_size/1024:.1f} KB)")


Note: you may need to restart the kernel to use updated packages.
Found 196 source file(s) in dataset/:

  [.MD  ] md files\chats\Nayari_chat_1.md  (4.9 KB)
  [.MD  ] md files\chats\Nayari_chat_10.md  (4.9 KB)
  [.MD  ] md files\chats\Nayari_chat_11.md  (4.8 KB)
  [.MD  ] md files\chats\Nayari_chat_12.md  (4.7 KB)
  [.MD  ] md files\chats\Nayari_chat_13.md  (4.6 KB)
  [.MD  ] md files\chats\Nayari_chat_14.md  (4.8 KB)
  [.MD  ] md files\chats\Nayari_chat_15.md  (1.2 KB)
  [.MD  ] md files\chats\Nayari_chat_16.md  (1.1 KB)
  [.MD  ] md files\chats\Nayari_chat_17.md  (1.2 KB)
  [.MD  ] md files\chats\Nayari_chat_18.md  (1.2 KB)
  [.MD  ] md files\chats\Nayari_chat_19.md  (1.4 KB)
  [.MD  ] md files\chats\Nayari_chat_2.md  (4.3 KB)
  [.MD  ] md files\chats\Nayari_chat_20.md  (1.0 KB)
  [.MD  ] md files\chats\Nayari_chat_21.md  (1.3 KB)
  [.MD  ] md files\chats\Nayari_chat_22.md  (1.2 KB)
  [.MD  ] md files\chats\Nayari_chat_23.md  (1.2 KB)
  [.MD  ] md files\chats\Nayari_chat_24.md  (1.1 

## 1 — Character Details

In [5]:
# Locate the details file anywhere inside dataset/
details_candidates = list(DATASET_DIR.rglob("*details*"))
if not details_candidates:
    print("⚠️  No details file found — character metadata will be empty.")
    char_name = char_type = char_gender = char_traits = char_personality = ""
else:
    details_text = details_candidates[0].read_text(encoding="utf-8", errors="replace")

    def extract_field(text, field):
        m = re.search(rf"\*\*{field}\*\*:?\s*(.+)", text)
        return m.group(1).strip() if m else ""

    char_name        = extract_field(details_text, "Name")
    char_type        = extract_field(details_text, "Type")
    char_gender      = extract_field(details_text, "Gender")
    char_traits      = extract_field(details_text, "Traits")
    char_personality = extract_field(details_text, "Personally")
    print(f"Character : {char_name} | {char_type} | {char_gender}")
    print(f"Details file: {details_candidates[0].relative_to(DATASET_DIR)}")

lore_sections = []


Character : Nayari | Kemonomimi | Female (Cis)
Details file: md files\lore\Nayari_Details.md


## 2 — Robust Multi-Format Parser

Handles every speaker format seen in the repo:

```
Name: text           ← plain
**Name**: text       ← bold markdown
> Name: text         ← blockquote
[Name]: text         ← bracket
```

Scene splitting triggers on: `--- END ---`, `=== break ===`, `<end>`, or **3+ blank lines**.
OOC annotations `[OOC: …]` and `(OOC …)` are stripped from message content.

In [6]:
# ── Core aliases (extended automatically per-file below) ────────────────────
BASE_USER_ALIASES      = {"me", "you", "tiaya"}
BASE_ASSISTANT_ALIASES = {"nayari", "nayri", "aura"}

# Universal speaker line pattern — covers all 4 formats
SPEAKER_RE = re.compile(
    r"""^
    (?:\*{1,2}|\[|>\s*)?          # optional: ** or [ or >
    ([A-Za-z][A-Za-z0-9 _\'\-]*)   # speaker name
    (?:\*{1,2}|\])?                 # optional closing ** or ]
    :\s*(.*)                         # colon + rest of line
    $""",
    re.VERBOSE,
)

END_RE = re.compile(
    r"^[-=*]{3,}\s*(<?(end|END|break|BREAK|scene|SCENE)>?)?\s*[-=*]{0,}$"
)

OOC_RE = re.compile(r"\[OOC:?[^\]]*\]|\(OOC[^)]*\)", re.IGNORECASE)


def _strip_ooc(text: str) -> str:
    return OOC_RE.sub("", text).strip()


def _inject_scene_ends(text: str) -> str:
    """Replace 3+ consecutive blank lines with a synthetic END marker."""
    return re.sub(r"(\r?\n){3,}", "\n--- END ---\n", text)


def _detect_aliases(text: str, filename: str):
    """
    Heuristically discover user/assistant aliases from the file.
    Any speaker that appears >= 2 times AND whose name contains a
    known assistant keyword is mapped to assistant; everything else
    that appears frequently becomes a user alias candidate.
    """
    counts = defaultdict(int)
    for line in text.splitlines():
        m = SPEAKER_RE.match(line.strip())
        if m:
            counts[m.group(1).strip().lower()] += 1

    user_extra, asst_extra = set(), set()
    asst_keywords = {"nayari", "nayri", "aura", "goddess"}
    for name, cnt in counts.items():
        if cnt < 2:
            continue
        if any(kw in name for kw in asst_keywords):
            asst_extra.add(name)
        else:
            user_extra.add(name)
    return user_extra, asst_extra


def parse_chat_text(text: str, filename: str = ""):
    """
    Full-featured chat parser.
    Returns list of {messages, source, turns} dicts.
    """
    user_aliases = BASE_USER_ALIASES.copy()
    asst_aliases = BASE_ASSISTANT_ALIASES.copy()
    u_extra, a_extra = _detect_aliases(text, filename)
    user_aliases |= u_extra
    asst_aliases |= a_extra

    text = _inject_scene_ends(text)
    lines = text.splitlines()
    conversations, current_messages = [], []
    current_role, buf = None, []
    skipped_lines = []

    def flush():
        nonlocal buf
        if current_role and buf:
            raw = " ".join(" ".join(buf).split()).strip()
            content = _strip_ooc(raw)
            if len(content) >= 10:          # quality: min 10 chars
                current_messages.append({"role": current_role, "content": content})
            elif content:
                skipped_lines.append(f"  ⚠ Short message ({len(content)} chars): {content!r}")
        buf = []

    def save():
        if len(current_messages) >= 2:      # quality: min 2 turns
            conversations.append({
                "messages": list(current_messages),
                "source": filename,
            })
        current_messages.clear()

    for line in lines:
        s = line.strip()
        if not s:
            continue
        # heading / metadata lines — skip
        if s.startswith(("##", "#", "---", "===")) and not SPEAKER_RE.match(s):
            if END_RE.match(s) or "<end>" in s.lower():
                flush(); save(); current_role = None
            continue
        if END_RE.match(s) or "<end>" in s.lower() or "<--- end" in s.lower():
            flush(); save(); current_role = None; continue

        m = SPEAKER_RE.match(s)
        if m:
            sp   = m.group(1).strip().lower()
            rest = m.group(2).strip()
            if sp in user_aliases:
                flush(); current_role = "user"; buf = [rest] if rest else []
            elif sp in asst_aliases:
                flush(); current_role = "assistant"; buf = [rest] if rest else []
            else:
                # unknown speaker — carry on accumulating if mid-turn
                if current_role:
                    buf.append(s)
        else:
            if current_role:
                buf.append(s)

    flush(); save()

    if skipped_lines:
        print(f"    [{filename}] quality skips:")
        for w in skipped_lines[:5]:
            print(w)

    return conversations


# ── deduplication ─────────────────────────────────────────────────────────────
def _fingerprint(conv: dict) -> str:
    norm = " ".join(
        m["content"][:80].lower()
        for m in conv["messages"]
    )
    return hashlib.md5(norm.encode()).hexdigest()


def deduplicate(convs: list) -> list:
    seen, out = set(), []
    for c in convs:
        fp = _fingerprint(c)
        if fp not in seen:
            seen.add(fp)
            out.append(c)
    return out


# ── PDF extractor ─────────────────────────────────────────────────────────────
def extract_pdf(path: Path):
    chat_convs, lore_chunks = [], []
    try:
        with pdfplumber.open(path) as pdf:
            print(f"  {path.name}: {len(pdf.pages)} page(s)")
            for i, page in enumerate(pdf.pages):
                text = page.extract_text() or ""
                if not text.strip():
                    print(f"    Page {i+1}: empty — skipped"); continue
                convs = parse_chat_text(text, path.name)
                if convs:
                    print(f"    Page {i+1}: CHAT  — {len(convs)} scene(s)")
                    chat_convs.extend(convs)
                else:
                    cleaned = re.sub(r"\n{3,}", "\n\n", text).strip()
                    lore_chunks.append(cleaned)
                    print(f"    Page {i+1}: LORE  — {len(cleaned)} chars")
    except Exception as exc:
        print(f"  ❌ Failed to read {path.name}: {exc}")
    return chat_convs, lore_chunks


def is_details_file(path: Path) -> bool:
    return "details" in path.name.lower() or "detail" in path.name.lower()


print("✅ Parser v3 ready — multi-format, auto-alias, OOC-strip, dedup.")


✅ Parser v3 ready — multi-format, auto-alias, OOC-strip, dedup.


## 3 — Parse All Source Files (.md / .txt / .pdf / .json)

In [7]:
md_conversations  = []
txt_conversations = []
pdf_conversations = []
json_conversations = []

chat_files = [f for f in all_files if not is_details_file(f)]
print(f"Processing {len(chat_files)} file(s)...\n")

for path in chat_files:
    ext = path.suffix.lower()
    rel = str(path.relative_to(DATASET_DIR))

    if ext in {".md", ".txt"}:
        try:
            text  = path.read_text(encoding="utf-8", errors="replace")
            convs = parse_chat_text(text, rel)
            label = "md" if ext == ".md" else "txt"
            print(f"  [{label.upper():4}] {rel}: {len(convs)} conversation(s)")
            if ext == ".md":
                md_conversations.extend(convs)
            else:
                txt_conversations.extend(convs)
        except Exception as exc:
            print(f"  ❌  {rel}: {exc}")

    elif ext == ".json":
        try:
            text = path.read_text(encoding="utf-8", errors="replace")
            data = json.loads(text)
            convs = []
            if isinstance(data, list):
                convs = [c for c in data if "messages" in c]
            elif isinstance(data, dict):
                if "conversations" in data:
                    convs = data["conversations"]
                elif "messages" in data:
                    convs = [data]
            
            for c in convs:
                if "source" not in c:
                    c["source"] = rel
            
            print(f"  [JSON] {rel}: {len(convs)} conversation(s)")
            json_conversations.extend(convs)
        except Exception as exc:
            print(f"  ❌  {rel}: {exc}")

    elif ext == ".pdf":
        print(f"  [PDF ] {path.name}")
        chat_convs, lore_chunks = extract_pdf(path)
        pdf_conversations.extend(chat_convs)
        if lore_chunks:
            name = path.stem.strip()
            text = "\n\n".join(lore_chunks)
            lore_sections.append({"source": name, "text": text})
            print(f"    → Lore '{name}' stored ({len(text)} chars)")
        print()

all_raw = md_conversations + txt_conversations + pdf_conversations + json_conversations
print(f"\nRaw conversations: {len(all_raw)}")
print(f"  {len(md_conversations)} md  |  {len(txt_conversations)} txt  |  {len(pdf_conversations)} pdf  |  {len(json_conversations)} json")


Processing 192 file(s)...

  [MD  ] md files\chats\Nayari_chat_1.md: 1 conversation(s)
    [md files\chats\Nayari_chat_10.md] quality skips:
  ⚠ Short message (5 chars): 'Rain?'
  [MD  ] md files\chats\Nayari_chat_10.md: 4 conversation(s)
  [MD  ] md files\chats\Nayari_chat_11.md: 4 conversation(s)
    [md files\chats\Nayari_chat_12.md] quality skips:
  ⚠ Short message (9 chars): 'Ballpark.'
  ⚠ Short message (5 chars): 'Yeah?'
  [MD  ] md files\chats\Nayari_chat_12.md: 4 conversation(s)
    [md files\chats\Nayari_chat_13.md] quality skips:
  ⚠ Short message (7 chars): 'And...?'
  [MD  ] md files\chats\Nayari_chat_13.md: 4 conversation(s)
  [MD  ] md files\chats\Nayari_chat_14.md: 4 conversation(s)
  [MD  ] md files\chats\Nayari_chat_15.md: 1 conversation(s)
  [MD  ] md files\chats\Nayari_chat_16.md: 1 conversation(s)
  [MD  ] md files\chats\Nayari_chat_17.md: 1 conversation(s)
  [MD  ] md files\chats\Nayari_chat_18.md: 1 conversation(s)
  [MD  ] md files\chats\Nayari_chat_19.md: 1 con

Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats


    Page 2: LORE  — 1568 chars
    → Lore 'Discovery' stored (4163 chars)

  [PDF ] Her Beliefs .pdf
  Her Beliefs .pdf: 2 page(s)
    Page 1: LORE  — 1408 chars
    Page 2: LORE  — 402 chars
    → Lore 'Her Beliefs' stored (1812 chars)


Raw conversations: 103
  82 md  |  0 txt  |  21 pdf  |  0 json


## 4 — Lore → Training Conversations

Each PDF lore section is chunked into paragraphs.
Each chunk is paired with a varied natural-language prompt so the model learns
to recall Nayari's lore organically. Multiple prompts per chunk for augmentation.

In [8]:
LORE_PROMPTS = [
    "Tell me about yourself.",
    "What's your story?",
    "Who are you, really?",
    "Share something about yourself with me.",
    "What do you believe in?",
    "Tell me what you believe.",
    "What matters most to you?",
    "What drives you?",
    "Describe yourself to me.",
    "How would you describe who you are?",
    "What makes you, you?",
    "Can you tell me more about your past?",
]

def lore_to_convs(sections, min_chars=150, augment=False):
    # Split each lore section into paragraph chunks.
    # augment=True pairs each chunk with TWO different prompts.
    convs = []
    prompt_idx = 0
    for section in sections:
        raw_chunks = [c.strip() for c in section["text"].split("\n\n") if c.strip()]
        chunks, buf = [], ""
        for chunk in raw_chunks:
            buf = (buf + "\n\n" + chunk).strip() if buf else chunk
            if len(buf) >= min_chars:
                chunks.append(buf); buf = ""
        if buf:
            if chunks: chunks[-1] += "\n\n" + buf
            else: chunks.append(buf)
        reps = 2 if augment else 1
        for chunk in chunks:
            for _ in range(reps):
                prompt = LORE_PROMPTS[prompt_idx % len(LORE_PROMPTS)]
                convs.append({
                    "messages": [
                        {"role": "user",      "content": prompt},
                        {"role": "assistant", "content": chunk},
                    ],
                    "source": section["source"],
                })
                prompt_idx += 1
    return convs

lore_conversations = lore_to_convs(lore_sections, augment=False)
print(f"Lore sections      : {len(lore_sections)}")
print(f"Lore training convs: {len(lore_conversations)}")
for i, c in enumerate(lore_conversations):
    q = c["messages"][0]["content"]
    a = c["messages"][1]["content"]
    print(f"  [{i+1:2}] Q: {q!r:<40}  A ({len(a)} chars): {a[:50]!r}...")

Lore sections      : 68
Lore training convs: 930
  [ 1] Q: 'Tell me about yourself.'                 A (223 chars): 'U\nNIT\nI\nG D\nEOGRAPHY AS A ISCIPLINE\nThis unit deal'...
  [ 2] Q: "What's your story?"                      A (3227 chars): 'CHAPTER\nG D\nEOGRAPHY AS A ISCIPLINE\nY ou have stud'...
  [ 3] Q: 'Who are you, really?'                    A (4362 chars): 'GEOGRAPHY AS A DISCIPLINE 3\nPut together, they mea'...
  [ 4] Q: 'Share something about yourself with me.'  A (4214 chars): '4 FUNDAMENTALS OF PHYSICAL GEOGRAPHY\ngradually, it'...
  [ 5] Q: 'What do you believe in?'                 A (3839 chars): 'GEOGRAPHY AS A DISCIPLINE 5\nYHPARGOEG\nFO\nDLEIF\nsen'...
  [ 6] Q: 'Tell me what you believe.'               A (745 chars): 'GEOGRAPHY AS A DISCIPLINE 7\nFigure 1.2 : Branches '...
  [ 7] Q: 'What matters most to you?'               A (2143 chars): '8 FUNDAMENTALS OF PHYSICAL GEOGRAPHY\nFigure 1.3 : '...
  [ 8] Q: 'What drives you?'                        A (3847 chars): 

## 5 — Deduplication & Quality Filter

In [9]:
all_conversations = all_raw + lore_conversations

before = len(all_conversations)
all_conversations = deduplicate(all_conversations)
after  = len(all_conversations)

print(f"Before dedup : {before}")
print(f"After  dedup : {after}  ({before - after} duplicates removed)")

# Sanity-check: flag any conversation with a very long single message
WARN_CHARS = 3000
flagged = [
    (i+1, m["role"], len(m["content"]))
    for i, c in enumerate(all_conversations)
    for m in c["messages"]
    if len(m["content"]) > WARN_CHARS
]
if flagged:
    print(f"\n⚠ {len(flagged)} message(s) exceed {WARN_CHARS} chars (may be lore, review if unexpected):")
    for idx, role, length in flagged[:8]:
        print(f"  Conv {idx} [{role}]: {length} chars")
else:
    print("\n✅ No oversized messages.")


Before dedup : 1033
After  dedup : 1031  (2 duplicates removed)

⚠ 91 message(s) exceed 3000 chars (may be lore, review if unexpected):
  Conv 105 [assistant]: 3227 chars
  Conv 106 [assistant]: 4362 chars
  Conv 107 [assistant]: 4214 chars
  Conv 108 [assistant]: 3839 chars
  Conv 111 [assistant]: 3847 chars
  Conv 117 [assistant]: 4209 chars
  Conv 120 [assistant]: 3161 chars
  Conv 121 [assistant]: 4165 chars


## 6 — Preview & Statistics Dashboard

In [10]:
from collections import Counter

# ── turn distribution ────────────────────────────────────────────────────────
turn_counts = [len(c["messages"]) for c in all_conversations]
total_msgs  = sum(turn_counts)
total_chars = sum(len(m["content"]) for c in all_conversations for m in c["messages"])
# rough token estimate: ~4 chars per token
token_est   = total_chars // 4

source_counts = Counter(c.get("source","?") for c in all_conversations)

print("=" * 60)
print(f"  DATASET STATISTICS")
print("=" * 60)
print(f"  Total conversations : {len(all_conversations)}")
print(f"    from markdown     : {len(md_conversations)}")
print(f"    from txt          : {len(txt_conversations)}")
print(f"    from pdf chats    : {len(pdf_conversations)}")
print(f"    from json         : {len(json_conversations)}")
print(f"    from lore         : {len(lore_conversations)}")
print(f"  Total messages      : {total_msgs}")
print(f"  Total characters    : {total_chars:,}")
print(f"  Est. tokens (÷4)    : {token_est:,}")
print(f"  Avg turns/conv      : {sum(turn_counts)/max(len(turn_counts),1):.1f}")
print(f"  Min / Max turns     : {min(turn_counts)} / {max(turn_counts)}")
print()
print("  Sources:")
for src, cnt in source_counts.most_common():
    print(f"    {src:<45} {cnt:>3} conv(s)")
print("=" * 60)

# ── conversation preview (first 3) ──────────────────────────────────────────
print()
for i, conv in enumerate(all_conversations[:3]):
    print(f"--- Conv {i+1} [{conv.get('source','?')}] ({len(conv['messages'])} turns) ---")
    for msg in conv["messages"]:
        label = "👤" if msg["role"] == "user" else "🌸"
        snippet = msg["content"][:90]
        ellipsis = "..." if len(msg["content"]) > 90 else ""
        print(f"  {label} {snippet}{ellipsis}")
    print()


  DATASET STATISTICS
  Total conversations : 1031
    from markdown     : 82
    from txt          : 0
    from pdf chats    : 21
    from json         : 0
    from lore         : 930
  Total messages      : 2955
  Total characters    : 2,073,057
  Est. tokens (÷4)    : 518,264
  Avg turns/conv      : 2.9
  Min / Max turns     : 2 / 26

  Sources:
    kehs102                                        47 conv(s)
    kehs107                                        34 conv(s)
    kehs106                                        29 conv(s)
    kehs101                                        27 conv(s)
    keps203                                        27 conv(s)
    kesy202                                        27 conv(s)
    keps201                                        25 conv(s)
    keps202                                        25 conv(s)
    keps206                                        25 conv(s)
    keps205                                        24 conv(s)
    keps209                   

## 7 — Export JSON

In [11]:
from datetime import datetime, timezone

dataset_json = {
    "meta": {
        "version": 3,
        "built_at": datetime.now(timezone.utc).isoformat(),
        "total_conversations": len(all_conversations),
        "sources": {
            "markdown": len(md_conversations),
            "txt":      len(txt_conversations),
            "pdf":      len(pdf_conversations),
            "json":     len(json_conversations),
            "lore":     len(lore_conversations),
        },
    },
    "character": {
        "name": char_name, "type": char_type, "gender": char_gender,
        "traits": char_traits, "personality": char_personality,
        "lore_sections": lore_sections,
    },
    "conversations": all_conversations,
}

OUTPUT_FILE.write_text(
    json.dumps(dataset_json, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
size_kb = OUTPUT_FILE.stat().st_size / 1024
print(f"✅ Saved → {OUTPUT_FILE.resolve()}")
print(f"   {len(all_conversations)} conversations | {size_kb:.1f} KB")


✅ Saved → T:\Documents\Github Desktop\Nayari-AI\nayari_dataset.json
   1031 conversations | 4303.7 KB


## 8 — Upload to Kaggle via API

**Before running:**
1. Go to [kaggle.com](https://kaggle.com) → Profile → **Settings** → **API** → **Create New Token**
2. A `kaggle.json` downloads — open it and copy the `username` and `key` values
3. Paste them into the two variables below

> The dataset is created as **private** by default.


In [ ]:
# ── FILL THESE IN ──────────────────────────────────────────────────────────
KAGGLE_USERNAME = "kaggle_username"   # from kaggle.json
KAGGLE_KEY      = "kaggle_key"        # from kaggle.json
DATASET_TITLE   = "nayari-dataset"   # slug used in the Kaggle URL
IS_PUBLIC       = False               # True to make the dataset public
# ───────────────────────────────────────────────────────────────────────────

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(exist_ok=True)
creds_file = kaggle_dir / "kaggle.json"
creds_file.write_text(
    json.dumps({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}),
    encoding="utf-8",
)
creds_file.chmod(0o600)
print("✅ Credentials written.")

STAGING = Path("_kaggle_upload")
shutil.rmtree(STAGING, ignore_errors=True)
STAGING.mkdir()
shutil.copy(OUTPUT_FILE, STAGING / OUTPUT_FILE.name)

(STAGING / "dataset-metadata.json").write_text(
    json.dumps({
        "title":     DATASET_TITLE,
        "id":        f"{KAGGLE_USERNAME}/{DATASET_TITLE}",
        "licenses":  [{"name": "CC0-1.0"}],
        "isPrivate": not IS_PUBLIC,
    }, indent=2),
    encoding="utf-8",
)
print(f"✅ Staging folder ready: {STAGING.resolve()}")

def run_kaggle(*args):
    return subprocess.run(["kaggle", *args], capture_output=True, text=True)

result  = run_kaggle("datasets", "create", "-p", str(STAGING), "--dir-mode", "zip")
combined = (result.stdout + result.stderr).lower()

if "already exists" in combined or "403" in combined:
    print("Dataset already exists — pushing a new version...")
    result = run_kaggle(
        "datasets", "version", "-p", str(STAGING),
        "-m", f"v3 build — {len(all_conversations)} conversations",
        "--dir-mode", "zip",
    )

print(result.stdout or result.stderr)

if result.returncode == 0:
    vis = "public" if IS_PUBLIC else "private"
    url = f"https://www.kaggle.com/datasets/{KAGGLE_USERNAME}/{DATASET_TITLE}"
    print(f"🎉 Upload successful! ({vis})")
    print(f"   Dataset URL: {url}")
    print()
    print("📋 Next steps in your Kaggle training notebook:")
    print("   1. Open nayari_train.ipynb on Kaggle")
    print(f"   2. Click '+ Add Data' → search '{DATASET_TITLE}' → Add")
    print("   3. Set Accelerator = GPU T4 x2, Internet = On → Run All")
else:
    print("❌ Upload failed. Double-check KAGGLE_USERNAME and KAGGLE_KEY.")

shutil.rmtree(STAGING, ignore_errors=True)
